# 10. 계절역전 검증 (②) — 표면유형·방사율로 판정

주장: 6월엔 그린필드(창릉·왕숙)가, 8월 폭염엔 도시(판교·분당)가 뜨겁다.
검증: 이게 **표면 물리(불투수 vs 저식생)** 때문인가, 아니면 나지 방사율 artifact인가?

- 격자별 6월LST(06-02·06-18 평균) vs 8월LST(08-05), Δ=8월−6월
- Δ를 imperv_frac·lowveg_frac와 상관 → 표면 구동 확인
- ST_EMIS(방사율) 표면유형별 비교 → artifact 배제

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, geopandas as gpd
from src.config import DATA_PROCESSED, OUT_FIGURES, CRS_LANDSAT, landsat_file
from src.raster.landsat import lst_celsius, read_tif
from src.raster.geo import read_geotransform, pixel_center_coords

grid = gpd.read_file(DATA_PROCESSED / "grid_features.geojson").to_crs(CRS_LANDSAT)
gt = read_geotransform(str(landsat_file("20240805", "ST_B10")))
xs, ys = pixel_center_coords(gt["gt"], gt["width"], gt["height"])

def to_grid(arr, name):
    v = ~np.isnan(arr)
    pts = gpd.GeoDataFrame({name: arr[v]}, geometry=gpd.points_from_xy(xs[v], ys[v]), crs=CRS_LANDSAT)
    j = gpd.sjoin(pts, grid[["grid_id", "geometry"]], predicate="within", how="inner")
    return j.groupby("grid_id")[name].mean()

jun = (lst_celsius("20240602") + lst_celsius("20240618")) / 2
g = grid.merge(to_grid(jun, "lst_jun"), on="grid_id", how="left")
g = g.merge(to_grid(lst_celsius("20240805"), "lst_aug"), on="grid_id", how="left")
# 방사율 (ST_EMIS scale 0.0001)
emis = read_tif(landsat_file("20240805", "ST_EMIS")).astype(np.float32) * 0.0001
emis[emis <= 0] = np.nan
g = g.merge(to_grid(emis, "emis"), on="grid_id", how="left")
g["delta"] = g["lst_aug"] - g["lst_jun"]
print("유효 격자:", g[["lst_jun", "lst_aug"]].dropna().shape[0])

유효 격자: 12552


In [2]:
d = g.dropna(subset=["lst_jun", "lst_aug", "imperv_frac"])
print("Δ(8월−6월) 상관:  imperv %.3f  |  lowveg %.3f  |  tree %.3f" % (
    d["delta"].corr(d["imperv_frac"]), d["delta"].corr(d["lowveg_frac"]), d["delta"].corr(d["tree_frac"])))
print("\n표면유형별 6월/8월/Δ LST (불투수 3분위):")
d = d.assign(imp_bin=pd.qcut(d["imperv_frac"], 3, labels=["저불투수", "중", "고불투수"]))
print(d.groupby("imp_bin")[["lst_jun", "lst_aug", "delta"]].mean().round(1))
print("\n방사율(ST_EMIS) 표면유형별 — artifact 검증:")
print("  고불투수:", round(d[d.imp_bin=='고불투수'].emis.mean(), 4),
      "| 저불투수(=저식생/나지):", round(d[d.imp_bin=='저불투수'].emis.mean(), 4))

Δ(8월−6월) 상관:  imperv 0.151  |  lowveg -0.159  |  tree -0.013

표면유형별 6월/8월/Δ LST (불투수 3분위):
           lst_jun    lst_aug  delta
imp_bin                             
저불투수     34.900002  37.400002    2.5
중        37.900002  41.599998    3.7
고불투수     38.700001  42.500000    3.8

방사율(ST_EMIS) 표면유형별 — artifact 검증:
  고불투수: 0.9624 | 저불투수(=저식생/나지): 0.9631


<프로젝트>\AppData\Local\Temp\ipykernel_57684\2163362296.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(d.groupby("imp_bin")[["lst_jun", "lst_aug", "delta"]].mean().round(1))


In [3]:
import matplotlib.pyplot as plt
from src.viz.fonts import setup_font
setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
sc = ax[0].scatter(d["lst_jun"], d["lst_aug"], c=d["imperv_frac"], s=2, cmap="pink_r", vmin=0, vmax=1)
lims = [d[["lst_jun", "lst_aug"]].min().min(), d[["lst_jun", "lst_aug"]].max().max()]
ax[0].plot(lims, lims, "k--", lw=0.8); ax[0].set_xlabel("6월 LST"); ax[0].set_ylabel("8월 LST")
ax[0].set_title("6월 vs 8월 (색=불투수율)"); plt.colorbar(sc, ax=ax[0], shrink=0.7)
for lab, sub in d.groupby("imp_bin"):
    ax[1].scatter(sub["lst_jun"].mean(), sub["lst_aug"].mean(), s=200, label=lab)
ax[1].plot(lims, lims, "k--", lw=0.8); ax[1].legend(); ax[1].set_xlabel("6월 LST"); ax[1].set_ylabel("8월 LST")
ax[1].set_title("표면유형 평균 이동")
plt.tight_layout(); plt.savefig(OUT_FIGURES / "fig13_seasonal.png", dpi=120); plt.close()
print("저장 fig13. 판정: Δ~imperv 양(+)·Δ~lowveg 음(−)이면 표면구동(진짜), 방사율 유사하면 artifact 아님")

<프로젝트>\AppData\Local\Temp\ipykernel_57684\3134022962.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for lab, sub in d.groupby("imp_bin"):


저장 fig13. 판정: Δ~imperv 양(+)·Δ~lowveg 음(−)이면 표면구동(진짜), 방사율 유사하면 artifact 아님
